# EA-TTM — Retail Shock Feature Engineering and Leakage Audit

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 06 — Recent-demand signals and future-known retail covariates  
**Scope:** Construct the two non-TTM information streams in the supervisor-approved shock-aware residual adapter and prove that they are available at the forecast origin.

This phase performs **no EA-TTM training** and does **not** inspect the locked final-test target.

## Adapter input contract

For forecast horizon $h$, the Retail Shock Encoder will later receive:

$$
\mathbf{u}_h
=
[
\mathbf{z}_t;
\mathbf{d}_h;
\mathbf{c}_h
].
$$

Phase 05 locked:

$$
\mathbf{d}_h
\in
\mathbb{R}^{128}.
$$

This phase defines:

$$
\mathbf{z}_t
\in
\mathbb{R}^{5}
$$

and:

$$
\mathbf{c}_h
\in
\mathbb{R}^{8}.
$$

Therefore the future Retail Shock Encoder input dimension is:

$$
5 + 128 + 8 = 141.
$$

## Recent-demand signal design

A 28-day recent-demand window is used because it matches the forecasting horizon while providing four complete weekly cycles.

Let:

$$
\mathcal{R}_t
=
\{y_{t-27},\ldots,y_t\}.
$$

The five raw signals are:

### Recent median

$$
m_t
=
\operatorname{median}(\mathcal{R}_t).
$$

### Robust scale

$$
s_t
=
1.4826
\operatorname{median}
\left(
|y_i-m_t|
\right).
$$

### Recent demand change

To reduce sensitivity to M5's high zero-demand rate, recent change compares two seven-day means:

$$
\Delta_t
=
\frac{1}{7}
\sum_{j=0}^{6}
y_{t-j}
-
\frac{1}{7}
\sum_{j=7}^{13}
y_{t-j}.
$$

### Short-term trend

The trend is the ordinary least-squares slope over the 28-day window:

$$
\beta_t
=
\frac{
\sum_{j=0}^{27}
(j-\bar j)(y_{t-27+j}-\bar y)
}{
\sum_{j=0}^{27}
(j-\bar j)^2
}.
$$

### Zero-demand rate

$$
\rho_t^{(0)}
=
\frac{1}{28}
\sum_{j=0}^{27}
\mathbb{1}
\left(
y_{t-j}=0
\right).
$$

Only observations at or before the forecast origin are used.

## Recent-feature stabilization

Retail demand is highly skewed. Before training-only standardization, the first four signals receive monotonic variance-stabilizing transforms:

$$
\tilde m_t
=
\log(1+m_t),
$$

$$
\tilde s_t
=
\log(1+s_t),
$$

$$
\widetilde{\Delta}_t
=
\operatorname{sign}(\Delta_t)
\log(1+|\Delta_t|),
$$

$$
\tilde\beta_t
=
\operatorname{sign}(\beta_t)
\log(1+|\beta_t|).
$$

The zero-demand rate is already bounded in $[0,1]$.

All five transformed signals are standardized using **training windows only**.

## Future-known retail covariate design

The adapter should focus on retail information that can plausibly modify the frozen TTM forecast rather than duplicate the base model's temporal representation.

For each horizon step $h$, the future-known covariate vector is:

$$
\mathbf{c}_h
=
[
\mathrm{SNAP}_h,\;
\mathrm{Event}_h,\;
A^{(p)}_h,\;
A^{(\Delta p)}_h,\;
\widetilde p_h,\;
\widetilde{\Delta p}_h,\;
\sin(\omega_h),\;
\cos(\omega_h)
].
$$

The eight components are:

1. state-specific SNAP indicator;
2. any-event indicator;
3. current-week price-record indicator;
4. price-change-valid indicator;
5. standardized $\log(1+\mathrm{price})$;
6. standardized weekly log-price change;
7. day-of-week sine;
8. day-of-week cosine.

The weekly log-price change is:

$$
\Delta p_h
=
\log
\left(
\frac{p_h}{p_{h-1}}
\right),
$$

when both current and previous weekly prices are observed. Otherwise it is set to zero and the corresponding validity indicator is zero.

The price-record indicators prevent missing price information from being silently treated as an ordinary observed zero.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

## Project preflight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")

PROCESSED_M5_DIR = PROJECT_ROOT / "data" / "processed" / "m5"
METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results" / "tables"

RUN_DIR = PROJECT_ROOT / "runs" / "06_retail_features"
FEATURE_DIR = RUN_DIR / "features"

for directory in (
    RESULTS_TABLE_DIR,
    RUN_DIR,
    FEATURE_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_artifacts = {
    "split_manifest": (
        METADATA_DIR
        / "02_SPLIT_MANIFEST.json"
    ),
    "phase05_summary": (
        METADATA_DIR
        / "05_RUN_SUMMARY.json"
    ),
    "phase05_cache_manifest": (
        METADATA_DIR
        / "05_CACHE_MANIFEST.json"
    ),
    "processed_sales": (
        PROCESSED_M5_DIR
        / "sales_train_evaluation.parquet"
    ),
    "processed_calendar": (
        PROCESSED_M5_DIR
        / "calendar.parquet"
    ),
    "processed_prices": (
        PROCESSED_M5_DIR
        / "sell_prices.parquet"
    ),
}

preflight = {
    name: {
        "path": str(path),
        "exists": path.exists(),
    }
    for name, path in required_artifacts.items()
}

missing_artifacts = [
    item["path"]
    for item in preflight.values()
    if not item["exists"]
]

assert not missing_artifacts, (
    "Phase 06 prerequisites are incomplete: "
    + "; ".join(missing_artifacts)
)

Mounted at /content/drive


## Recover locked project state

Phase 06 consumes the exact temporal and cache ordering established upstream.

Persisted metadata and the user-facing RETURN BLOCK are treated as separate schemas. Sample counts are recovered from the durable Phase 05 cache manifest rather than assuming that fields printed in the Phase 05 RETURN BLOCK were also written into `05_RUN_SUMMARY.json`.

In [3]:
split_manifest = json.loads(
    (
        METADATA_DIR
        / "02_SPLIT_MANIFEST.json"
    ).read_text(
        encoding="utf-8"
    )
)

phase05_summary = json.loads(
    (
        METADATA_DIR
        / "05_RUN_SUMMARY.json"
    ).read_text(
        encoding="utf-8"
    )
)

cache_manifest = json.loads(
    (
        METADATA_DIR
        / "05_CACHE_MANIFEST.json"
    ).read_text(
        encoding="utf-8"
    )
)

# Record schemas explicitly before consuming persisted artifacts.
phase05_summary_keys = sorted(
    phase05_summary.keys()
)

cache_manifest_keys = sorted(
    cache_manifest.keys()
)

required_phase05_keys = {
    "dataset_sha256",
    "split_sha256",
    "crossfeed_contract_sha256",
    "cache_contract_sha256",
    "horizon_aligned_decoder_shape",
}

missing_phase05_keys = sorted(
    required_phase05_keys
    - set(phase05_summary)
)

assert not missing_phase05_keys, (
    "Phase 05 run summary is missing required fields: "
    + ", ".join(missing_phase05_keys)
)

required_cache_manifest_keys = {
    "cache_contract_sha256",
    "sample_order",
    "train_decoder_hidden",
    "train_base_forecast",
    "validation_decoder_hidden",
    "validation_base_forecast",
}

missing_cache_manifest_keys = sorted(
    required_cache_manifest_keys
    - set(cache_manifest)
)

assert not missing_cache_manifest_keys, (
    "Phase 05 cache manifest is missing required fields: "
    + ", ".join(missing_cache_manifest_keys)
)

DATASET_SHA256 = (
    phase05_summary["dataset_sha256"]
)

SPLIT_SHA256 = (
    phase05_summary["split_sha256"]
)

CROSSFEED_CONTRACT_SHA256 = (
    phase05_summary[
        "crossfeed_contract_sha256"
    ]
)

CACHE_CONTRACT_SHA256 = (
    phase05_summary[
        "cache_contract_sha256"
    ]
)

CONTEXT_LENGTH = int(
    split_manifest["context_length"]
)

FORECAST_HORIZON = int(
    split_manifest["forecast_horizon"]
)

TRAIN_ORIGINS = [
    int(origin)
    for origin in split_manifest["train_origins"]
]

VALIDATION_ORIGIN = int(
    split_manifest["validation_origin"]
)

# The earlier Phase 06 notebook incorrectly expected `cache_counts`
# inside 05_RUN_SUMMARY.json. That field was printed in the Phase 05
# RETURN BLOCK, but was not persisted in the run-summary JSON.
#
# Recover sample counts from the durable cache manifest instead.
training_decoder_shape = (
    cache_manifest[
        "train_decoder_hidden"
    ]["shape"]
)

validation_decoder_shape = (
    cache_manifest[
        "validation_decoder_hidden"
    ]["shape"]
)

training_forecast_shape = (
    cache_manifest[
        "train_base_forecast"
    ]["shape"]
)

validation_forecast_shape = (
    cache_manifest[
        "validation_base_forecast"
    ]["shape"]
)

TRAINING_SAMPLE_COUNT = int(
    training_decoder_shape[0]
)

SERIES_COUNT = int(
    validation_decoder_shape[0]
)

assert DATASET_SHA256
assert SPLIT_SHA256 == split_manifest["split_sha256"]
assert (
    CACHE_CONTRACT_SHA256
    == cache_manifest["cache_contract_sha256"]
)

assert CONTEXT_LENGTH == 512
assert FORECAST_HORIZON == 28
assert TRAIN_ORIGINS == [512, 960, 1409, 1857]
assert VALIDATION_ORIGIN == 1885

assert TRAINING_SAMPLE_COUNT == 121_960
assert SERIES_COUNT == 30_490

assert training_decoder_shape == [
    121_960,
    1,
    9,
    128,
]

assert validation_decoder_shape == [
    30_490,
    1,
    9,
    128,
]

assert training_forecast_shape == [
    121_960,
    28,
]

assert validation_forecast_shape == [
    30_490,
    28,
]

assert (
    phase05_summary[
        "horizon_aligned_decoder_shape"
    ]
    == ["B", 28, 128]
)

assert (
    cache_manifest["sample_order"]["train"]
    == "origin-major then M5 store-item series order"
)

assert (
    cache_manifest["sample_order"]["validation"]
    == "M5 store-item series order"
)

assert (
    split_manifest["locked_test_interval"]
    == ["d_1914", "d_1941"]
)

print(
    "Phase 05 run-summary keys:",
    phase05_summary_keys,
)

print(
    "Phase 05 cache-manifest keys:",
    cache_manifest_keys,
)

print(
    "Recovered training sample count:",
    TRAINING_SAMPLE_COUNT,
)

print(
    "Recovered validation series count:",
    SERIES_COUNT,
)

Phase 05 run-summary keys: ['additional_trainable_parameters', 'cache_contract_sha256', 'cache_runtime_seconds', 'cached_validation_metrics', 'crossfeed_contract_sha256', 'dataset_sha256', 'horizon_aligned_decoder_shape', 'loaded_hp_ttm_total_parameters', 'loaded_hp_ttm_trainable_parameters', 'matched_native_indices_zero_based', 'raw_decoder_shape', 'reconstruction_max_abs_error', 'reconstruction_mean_abs_error', 'split_sha256', 'status']
Phase 05 cache-manifest keys: ['cache_contract_sha256', 'dataset_sha256', 'sample_order', 'split_sha256', 'train_base_forecast', 'train_decoder_hidden', 'validation_base_forecast', 'validation_decoder_hidden']
Recovered training sample count: 121960
Recovered validation series count: 30490


## Load development data only

The sales frame stops at the validation target end, `d_1913`. No final-test demand is loaded.

In [4]:
identifier_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

development_day_columns = [
    f"d_{day}"
    for day in range(
        1,
        VALIDATION_ORIGIN
        + FORECAST_HORIZON
        + 1,
    )
]

sales_frame = pd.read_parquet(
    (
        PROCESSED_M5_DIR
        / "sales_train_evaluation.parquet"
    ),
    columns=(
        identifier_columns
        + development_day_columns
    ),
)

development_values = sales_frame[
    development_day_columns
].to_numpy(
    dtype=np.float32,
    copy=True,
)

assert development_values.shape == (
    30_490,
    1913,
)

## Vectorized recent-demand signals

In [5]:
RECENT_LOOKBACK = 28
WEEK_LENGTH = 7

recent_x = np.arange(
    RECENT_LOOKBACK,
    dtype=np.float64,
)

recent_x_centered = (
    recent_x
    - recent_x.mean()
)

trend_denominator = float(
    np.square(
        recent_x_centered
    ).sum()
)

def compute_recent_raw_features(
    context_matrix: np.ndarray,
) -> np.ndarray:
    recent_window = context_matrix[
        :,
        -RECENT_LOOKBACK:,
    ].astype(
        np.float64,
        copy=False,
    )

    recent_median = np.median(
        recent_window,
        axis=1,
    )

    absolute_deviation = np.abs(
        recent_window
        - recent_median[:, None]
    )

    robust_scale = (
        1.4826
        * np.median(
            absolute_deviation,
            axis=1,
        )
    )

    recent_change = (
        recent_window[:, -7:].mean(
            axis=1
        )
        - recent_window[:, -14:-7].mean(
            axis=1
        )
    )

    recent_centered = (
        recent_window
        - recent_window.mean(
            axis=1,
            keepdims=True,
        )
    )

    short_term_trend = (
        recent_centered
        * recent_x_centered[None, :]
    ).sum(
        axis=1
    ) / trend_denominator

    zero_demand_rate = (
        recent_window == 0.0
    ).mean(
        axis=1
    )

    feature_matrix = np.column_stack(
        [
            recent_median,
            robust_scale,
            recent_change,
            short_term_trend,
            zero_demand_rate,
        ]
    ).astype(
        np.float32,
        copy=False,
    )

    assert np.isfinite(
        feature_matrix
    ).all()

    return feature_matrix

def stabilize_recent_features(
    raw_features: np.ndarray,
) -> np.ndarray:
    stabilized = np.empty_like(
        raw_features,
        dtype=np.float32,
    )

    stabilized[:, 0] = np.log1p(
        raw_features[:, 0]
    )

    stabilized[:, 1] = np.log1p(
        raw_features[:, 1]
    )

    stabilized[:, 2] = (
        np.sign(
            raw_features[:, 2]
        )
        * np.log1p(
            np.abs(
                raw_features[:, 2]
            )
        )
    )

    stabilized[:, 3] = (
        np.sign(
            raw_features[:, 3]
        )
        * np.log1p(
            np.abs(
                raw_features[:, 3]
            )
        )
    )

    stabilized[:, 4] = (
        raw_features[:, 4]
    )

    return stabilized

In [6]:
training_recent_raw = np.empty(
    (
        SERIES_COUNT
        * len(TRAIN_ORIGINS),
        5,
    ),
    dtype=np.float32,
)

for origin_index, origin in enumerate(
    TRAIN_ORIGINS
):
    destination_start = (
        origin_index
        * SERIES_COUNT
    )

    destination_end = (
        destination_start
        + SERIES_COUNT
    )

    origin_context = (
        development_values[
            :,
            origin - CONTEXT_LENGTH:origin,
        ]
    )

    training_recent_raw[
        destination_start:destination_end
    ] = compute_recent_raw_features(
        origin_context
    )

validation_context = development_values[
    :,
    VALIDATION_ORIGIN
    - CONTEXT_LENGTH:VALIDATION_ORIGIN,
]

validation_recent_raw = (
    compute_recent_raw_features(
        validation_context
    )
)

assert training_recent_raw.shape == (
    121_960,
    5,
)

assert validation_recent_raw.shape == (
    30_490,
    5,
)

## Training-only recent-feature standardization

No validation statistic is used to fit the feature transformation.

In [7]:
training_recent_stabilized = (
    stabilize_recent_features(
        training_recent_raw
    )
)

validation_recent_stabilized = (
    stabilize_recent_features(
        validation_recent_raw
    )
)

recent_feature_mean = (
    training_recent_stabilized
    .mean(
        axis=0,
        dtype=np.float64,
    )
)

recent_feature_std = (
    training_recent_stabilized
    .std(
        axis=0,
        dtype=np.float64,
    )
)

RECENT_STD_EPSILON = 1e-8

recent_feature_std = np.where(
    recent_feature_std
    < RECENT_STD_EPSILON,
    1.0,
    recent_feature_std,
)

training_recent_features = (
    (
        training_recent_stabilized
        - recent_feature_mean
    )
    / recent_feature_std
).astype(
    np.float32,
)

validation_recent_features = (
    (
        validation_recent_stabilized
        - recent_feature_mean
    )
    / recent_feature_std
).astype(
    np.float32,
)

assert np.isfinite(
    training_recent_features
).all()

assert np.isfinite(
    validation_recent_features
).all()

## Calendar and price schedule preparation

Calendar covariates are indexed by exact future target day. Price covariates are indexed by store, item, and Walmart week.

In [8]:
calendar_columns = [
    "d",
    "date",
    "wm_yr_wk",
    "wday",
    "event_name_1",
    "event_name_2",
    "snap_CA",
    "snap_TX",
    "snap_WI",
]

calendar = pd.read_parquet(
    (
        PROCESSED_M5_DIR
        / "calendar.parquet"
    ),
    columns=calendar_columns,
)

calendar = (
    calendar
    .assign(
        day_index=lambda frame: (
            frame["d"]
            .str.removeprefix("d_")
            .astype("int32")
        )
    )
    .sort_values(
        "day_index",
        kind="stable",
    )
)

calendar_by_day = (
    calendar
    .set_index("day_index")
)

assert all(
    day in calendar_by_day.index
    for day in range(
        1,
        VALIDATION_ORIGIN
        + FORECAST_HORIZON
        + 1,
    )
)

state_ids = sales_frame[
    "state_id"
].astype(str).to_numpy()

store_ids = sales_frame[
    "store_id"
].astype(str).to_numpy()

item_ids = sales_frame[
    "item_id"
].astype(str).to_numpy()

state_to_index = {
    "CA": 0,
    "TX": 1,
    "WI": 2,
}

state_indices = np.asarray(
    [
        state_to_index[state]
        for state in state_ids
    ],
    dtype=np.int8,
)

assert set(
    np.unique(state_ids)
) == {"CA", "TX", "WI"}

## Relevant price-week matrix

Only Walmart weeks required by the training and validation forecast horizons, plus their immediately preceding weeks, are loaded into the feature matrix.

In [9]:
all_feature_origins = (
    TRAIN_ORIGINS
    + [VALIDATION_ORIGIN]
)

target_days_by_origin: dict[int, list[int]] = {
    origin: list(
        range(
            origin + 1,
            origin + FORECAST_HORIZON + 1,
        )
    )
    for origin in all_feature_origins
}

target_weeks = sorted(
    {
        int(
            calendar_by_day.loc[
                target_day,
                "wm_yr_wk",
            ]
        )
        for target_days in (
            target_days_by_origin.values()
        )
        for target_day in target_days
    }
)

calendar_week_order = (
    calendar[
        ["wm_yr_wk", "date"]
    ]
    .drop_duplicates(
        "wm_yr_wk"
    )
    .sort_values(
        "date",
        kind="stable",
    )[
        "wm_yr_wk"
    ]
    .astype(int)
    .tolist()
)

previous_week_map: dict[int, int | None] = {}

for week_position, week_value in enumerate(
    calendar_week_order
):
    previous_week_map[
        week_value
    ] = (
        calendar_week_order[
            week_position - 1
        ]
        if week_position > 0
        else None
    )

previous_target_weeks = {
    previous_week_map[week]
    for week in target_weeks
    if previous_week_map[week]
    is not None
}

required_price_weeks = sorted(
    set(target_weeks)
    | previous_target_weeks
)

price_table = pd.read_parquet(
    (
        PROCESSED_M5_DIR
        / "sell_prices.parquet"
    ),
    columns=[
        "store_id",
        "item_id",
        "wm_yr_wk",
        "sell_price",
    ],
)

price_table = price_table.loc[
    price_table[
        "wm_yr_wk"
    ].isin(
        required_price_weeks
    )
].copy()

series_price_index = pd.MultiIndex.from_arrays(
    [
        store_ids,
        item_ids,
    ],
    names=[
        "store_id",
        "item_id",
    ],
)

price_pivot = (
    price_table
    .pivot(
        index=[
            "store_id",
            "item_id",
        ],
        columns="wm_yr_wk",
        values="sell_price",
    )
    .reindex(
        series_price_index
    )
)

for week in required_price_weeks:
    if week not in price_pivot.columns:
        price_pivot[week] = np.nan

price_pivot = price_pivot[
    required_price_weeks
]

price_matrix = price_pivot.to_numpy(
    dtype=np.float32,
    copy=True,
)

week_to_column_index = {
    int(week): index
    for index, week in enumerate(
        required_price_weeks
    )
}

assert price_matrix.shape[0] == SERIES_COUNT

## Future-known covariate constructor

In [10]:
FUTURE_COVARIATE_NAMES = [
    "snap_indicator",
    "event_indicator",
    "price_record_observed",
    "price_change_valid",
    "log_price",
    "weekly_log_price_change",
    "day_of_week_sin",
    "day_of_week_cos",
]

FUTURE_COVARIATE_DIM = len(
    FUTURE_COVARIATE_NAMES
)

assert FUTURE_COVARIATE_DIM == 8

def build_future_covariates_raw(
    origin: int,
) -> np.ndarray:
    output = np.empty(
        (
            SERIES_COUNT,
            FORECAST_HORIZON,
            FUTURE_COVARIATE_DIM,
        ),
        dtype=np.float32,
    )

    target_days = (
        target_days_by_origin[
            origin
        ]
    )

    for horizon_index, target_day in enumerate(
        target_days
    ):
        calendar_row = (
            calendar_by_day.loc[
                target_day
            ]
        )

        snap_by_state = np.asarray(
            [
                calendar_row["snap_CA"],
                calendar_row["snap_TX"],
                calendar_row["snap_WI"],
            ],
            dtype=np.float32,
        )

        snap_indicator = (
            snap_by_state[
                state_indices
            ]
        )

        event_indicator = np.float32(
            (
                pd.notna(
                    calendar_row[
                        "event_name_1"
                    ]
                )
            )
            or (
                pd.notna(
                    calendar_row[
                        "event_name_2"
                    ]
                )
            )
        )

        current_week = int(
            calendar_row[
                "wm_yr_wk"
            ]
        )

        previous_week = (
            previous_week_map[
                current_week
            ]
        )

        current_price = (
            price_matrix[
                :,
                week_to_column_index[
                    current_week
                ],
            ]
        )

        current_observed = (
            np.isfinite(
                current_price
            )
            & (
                current_price > 0.0
            )
        )

        if (
            previous_week is not None
            and previous_week
            in week_to_column_index
        ):
            previous_price = (
                price_matrix[
                    :,
                    week_to_column_index[
                        previous_week
                    ],
                ]
            )

            previous_observed = (
                np.isfinite(
                    previous_price
                )
                & (
                    previous_price > 0.0
                )
            )
        else:
            previous_price = np.full(
                SERIES_COUNT,
                np.nan,
                dtype=np.float32,
            )

            previous_observed = np.zeros(
                SERIES_COUNT,
                dtype=bool,
            )

        price_change_valid = (
            current_observed
            & previous_observed
        )

        log_price = np.zeros(
            SERIES_COUNT,
            dtype=np.float32,
        )

        log_price[
            current_observed
        ] = np.log1p(
            current_price[
                current_observed
            ]
        ).astype(
            np.float32
        )

        log_price_change = np.zeros(
            SERIES_COUNT,
            dtype=np.float32,
        )

        log_price_change[
            price_change_valid
        ] = np.log(
            current_price[
                price_change_valid
            ]
            / previous_price[
                price_change_valid
            ]
        ).astype(
            np.float32
        )

        wday = int(
            calendar_row[
                "wday"
            ]
        )

        angle = (
            2.0
            * math.pi
            * (wday - 1)
            / 7.0
        )

        output[
            :,
            horizon_index,
            0,
        ] = snap_indicator

        output[
            :,
            horizon_index,
            1,
        ] = event_indicator

        output[
            :,
            horizon_index,
            2,
        ] = current_observed.astype(
            np.float32
        )

        output[
            :,
            horizon_index,
            3,
        ] = price_change_valid.astype(
            np.float32
        )

        output[
            :,
            horizon_index,
            4,
        ] = log_price

        output[
            :,
            horizon_index,
            5,
        ] = log_price_change

        output[
            :,
            horizon_index,
            6,
        ] = np.float32(
            math.sin(angle)
        )

        output[
            :,
            horizon_index,
            7,
        ] = np.float32(
            math.cos(angle)
        )

    assert np.isfinite(
        output
    ).all()

    return output

In [11]:
training_future_raw = np.empty(
    (
        SERIES_COUNT
        * len(TRAIN_ORIGINS),
        FORECAST_HORIZON,
        FUTURE_COVARIATE_DIM,
    ),
    dtype=np.float32,
)

for origin_index, origin in enumerate(
    TRAIN_ORIGINS
):
    destination_start = (
        origin_index
        * SERIES_COUNT
    )

    destination_end = (
        destination_start
        + SERIES_COUNT
    )

    training_future_raw[
        destination_start:destination_end
    ] = build_future_covariates_raw(
        origin
    )

validation_future_raw = (
    build_future_covariates_raw(
        VALIDATION_ORIGIN
    )
)

assert training_future_raw.shape == (
    121_960,
    28,
    8,
)

assert validation_future_raw.shape == (
    30_490,
    28,
    8,
)

## Training-only price scaling

Binary indicators and cyclic variables remain in their natural ranges. Only the two continuous price features are standardized.

Missing/unavailable price values are kept at standardized zero, with their corresponding validity indicators identifying their status.

In [12]:
PRICE_FEATURE_INDEX = 4
PRICE_CHANGE_FEATURE_INDEX = 5
PRICE_OBSERVED_INDEX = 2
PRICE_CHANGE_VALID_INDEX = 3

training_price_observed_mask = (
    training_future_raw[
        :,
        :,
        PRICE_OBSERVED_INDEX,
    ] == 1.0
)

training_change_valid_mask = (
    training_future_raw[
        :,
        :,
        PRICE_CHANGE_VALID_INDEX,
    ] == 1.0
)

training_log_price_values = (
    training_future_raw[
        :,
        :,
        PRICE_FEATURE_INDEX,
    ][
        training_price_observed_mask
    ]
)

training_log_change_values = (
    training_future_raw[
        :,
        :,
        PRICE_CHANGE_FEATURE_INDEX,
    ][
        training_change_valid_mask
    ]
)

assert (
    training_log_price_values.size
    > 0
)

assert (
    training_log_change_values.size
    > 0
)

price_mean = float(
    training_log_price_values.mean(
        dtype=np.float64
    )
)

price_std = float(
    training_log_price_values.std(
        dtype=np.float64
    )
)

price_change_mean = float(
    training_log_change_values.mean(
        dtype=np.float64
    )
)

price_change_std = float(
    training_log_change_values.std(
        dtype=np.float64
    )
)

if price_std < 1e-8:
    price_std = 1.0

if price_change_std < 1e-8:
    price_change_std = 1.0

def scale_future_covariates(
    raw_covariates: np.ndarray,
) -> np.ndarray:
    scaled = raw_covariates.copy()

    price_observed = (
        scaled[
            :,
            :,
            PRICE_OBSERVED_INDEX,
        ] == 1.0
    )

    price_change_valid = (
        scaled[
            :,
            :,
            PRICE_CHANGE_VALID_INDEX,
        ] == 1.0
    )

    scaled[
        :,
        :,
        PRICE_FEATURE_INDEX,
    ] = 0.0

    scaled[
        :,
        :,
        PRICE_CHANGE_FEATURE_INDEX,
    ] = 0.0

    raw_price = raw_covariates[
        :,
        :,
        PRICE_FEATURE_INDEX,
    ]

    raw_change = raw_covariates[
        :,
        :,
        PRICE_CHANGE_FEATURE_INDEX,
    ]

    scaled[
        :,
        :,
        PRICE_FEATURE_INDEX,
    ][
        price_observed
    ] = (
        (
            raw_price[
                price_observed
            ]
            - price_mean
        )
        / price_std
    )

    scaled[
        :,
        :,
        PRICE_CHANGE_FEATURE_INDEX,
    ][
        price_change_valid
    ] = (
        (
            raw_change[
                price_change_valid
            ]
            - price_change_mean
        )
        / price_change_std
    )

    return scaled.astype(
        np.float32,
        copy=False,
    )

training_future_covariates = (
    scale_future_covariates(
        training_future_raw
    )
)

validation_future_covariates = (
    scale_future_covariates(
        validation_future_raw
    )
)

assert np.isfinite(
    training_future_covariates
).all()

assert np.isfinite(
    validation_future_covariates
).all()

## Leakage audit

For every sample with forecast origin $o$:

- recent-demand signals use only $d_{o-27}:d_o$;
- future calendar/SNAP/event covariates use $d_{o+1}:d_{o+28}$;
- future price features use the supplied weekly price schedule for those horizon dates and the immediately preceding weekly price;
- no future demand value is used to construct $\mathbf{z}_t$ or $\mathbf{c}_h$;
- all learned scaling statistics are fitted only on training windows.

The final-test demand interval remains unmaterialized.

In [13]:
leakage_records: list[dict[str, Any]] = []

for role, origins in (
    ("train", TRAIN_ORIGINS),
    ("validation", [VALIDATION_ORIGIN]),
):
    for origin in origins:
        recent_start = (
            origin
            - RECENT_LOOKBACK
            + 1
        )

        recent_end = origin
        future_start = origin + 1
        future_end = (
            origin
            + FORECAST_HORIZON
        )

        leakage_records.append(
            {
                "role": role,
                "origin": origin,
                "recent_demand_start": (
                    recent_start
                ),
                "recent_demand_end": (
                    recent_end
                ),
                "future_covariate_start": (
                    future_start
                ),
                "future_covariate_end": (
                    future_end
                ),
                "recent_demand_uses_future_target": (
                    recent_end > origin
                ),
                "future_covariates_include_demand": False,
                "scaler_fit_scope": (
                    "training_only"
                ),
            }
        )

leakage_table = pd.DataFrame(
    leakage_records
)

assert not leakage_table[
    "recent_demand_uses_future_target"
].any()

assert (
    leakage_table[
        "future_covariates_include_demand"
    ] == False
).all()

assert (
    leakage_table.loc[
        leakage_table[
            "role"
        ] == "validation",
        "future_covariate_end",
    ].max()
    == 1913
)

assert (
    "d_1914"
    not in development_day_columns
)

## Feature diagnostics

In [14]:
RECENT_FEATURE_NAMES = [
    "recent_median",
    "robust_scale_mad",
    "recent_7d_mean_change",
    "short_term_28d_trend",
    "zero_demand_rate",
]

feature_summary_records: list[
    dict[str, Any]
] = []

for feature_index, feature_name in enumerate(
    RECENT_FEATURE_NAMES
):
    values = training_recent_features[
        :,
        feature_index,
    ]

    feature_summary_records.append(
        {
            "feature_group": "recent_demand",
            "feature_name": feature_name,
            "mean": float(
                values.mean(
                    dtype=np.float64
                )
            ),
            "std": float(
                values.std(
                    dtype=np.float64
                )
            ),
            "min": float(
                values.min()
            ),
            "max": float(
                values.max()
            ),
        }
    )

for feature_index, feature_name in enumerate(
    FUTURE_COVARIATE_NAMES
):
    values = training_future_covariates[
        :,
        :,
        feature_index,
    ]

    feature_summary_records.append(
        {
            "feature_group": "future_covariate",
            "feature_name": feature_name,
            "mean": float(
                values.mean(
                    dtype=np.float64
                )
            ),
            "std": float(
                values.std(
                    dtype=np.float64
                )
            ),
            "min": float(
                values.min()
            ),
            "max": float(
                values.max()
            ),
        }
    )

feature_summary = pd.DataFrame(
    feature_summary_records
)

price_record_observed_rate = float(
    training_future_raw[
        :,
        :,
        PRICE_OBSERVED_INDEX,
    ].mean(
        dtype=np.float64
    )
)

price_change_valid_rate = float(
    training_future_raw[
        :,
        :,
        PRICE_CHANGE_VALID_INDEX,
    ].mean(
        dtype=np.float64
    )
)

snap_rate = float(
    training_future_raw[
        :,
        :,
        0,
    ].mean(
        dtype=np.float64
    )
)

event_rate = float(
    training_future_raw[
        :,
        :,
        1,
    ].mean(
        dtype=np.float64
    )
)

## Persist model-ready feature artifacts

In [15]:
TRAIN_RECENT_PATH = (
    FEATURE_DIR
    / "train_recent_demand_features.npy"
)

VALIDATION_RECENT_PATH = (
    FEATURE_DIR
    / "validation_recent_demand_features.npy"
)

TRAIN_FUTURE_PATH = (
    FEATURE_DIR
    / "train_future_covariates.npy"
)

VALIDATION_FUTURE_PATH = (
    FEATURE_DIR
    / "validation_future_covariates.npy"
)

FEATURE_SCALER_PATH = (
    METADATA_DIR
    / "06_FEATURE_SCALER.json"
)

FEATURE_CONTRACT_PATH = (
    METADATA_DIR
    / "06_FEATURE_CONTRACT.json"
)

LEAKAGE_AUDIT_PATH = (
    RESULTS_TABLE_DIR
    / "06_LEAKAGE_AUDIT.csv"
)

FEATURE_SUMMARY_PATH = (
    RESULTS_TABLE_DIR
    / "06_FEATURE_SUMMARY.csv"
)

RUN_SUMMARY_PATH = (
    METADATA_DIR
    / "06_RUN_SUMMARY.json"
)

np.save(
    TRAIN_RECENT_PATH,
    training_recent_features,
)

np.save(
    VALIDATION_RECENT_PATH,
    validation_recent_features,
)

np.save(
    TRAIN_FUTURE_PATH,
    training_future_covariates,
)

np.save(
    VALIDATION_FUTURE_PATH,
    validation_future_covariates,
)

leakage_table.to_csv(
    LEAKAGE_AUDIT_PATH,
    index=False,
)

feature_summary.to_csv(
    FEATURE_SUMMARY_PATH,
    index=False,
)

feature_scaler = {
    "recent_feature_names": (
        RECENT_FEATURE_NAMES
    ),
    "recent_feature_mean": (
        recent_feature_mean.tolist()
    ),
    "recent_feature_std": (
        recent_feature_std.tolist()
    ),
    "recent_stabilization": {
        "recent_median": "log1p",
        "robust_scale_mad": "log1p",
        "recent_7d_mean_change": (
            "signed_log1p"
        ),
        "short_term_28d_trend": (
            "signed_log1p"
        ),
        "zero_demand_rate": "identity",
    },
    "future_covariate_names": (
        FUTURE_COVARIATE_NAMES
    ),
    "continuous_future_scaling": {
        "log_price": {
            "mean": price_mean,
            "std": price_std,
            "fit_scope": (
                "observed training prices only"
            ),
        },
        "weekly_log_price_change": {
            "mean": price_change_mean,
            "std": price_change_std,
            "fit_scope": (
                "valid training price changes only"
            ),
        },
    },
}

FEATURE_SCALER_PATH.write_text(
    json.dumps(
        feature_scaler,
        indent=2,
    ),
    encoding="utf-8",
)

feature_contract = {
    "status": (
        "PHASE_06_FEATURE_CONTRACT_LOCKED"
    ),
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "crossfeed_contract_sha256": (
        CROSSFEED_CONTRACT_SHA256
    ),
    "cache_contract_sha256": (
        CACHE_CONTRACT_SHA256
    ),
    "recent_lookback_days": (
        RECENT_LOOKBACK
    ),
    "recent_feature_names": (
        RECENT_FEATURE_NAMES
    ),
    "recent_feature_dimension": 5,
    "decoder_feature_dimension": 128,
    "future_covariate_names": (
        FUTURE_COVARIATE_NAMES
    ),
    "future_covariate_dimension": 8,
    "retail_shock_encoder_input_dimension": (
        5 + 128 + 8
    ),
    "training_feature_shape": {
        "recent": list(
            training_recent_features.shape
        ),
        "future": list(
            training_future_covariates.shape
        ),
    },
    "validation_feature_shape": {
        "recent": list(
            validation_recent_features.shape
        ),
        "future": list(
            validation_future_covariates.shape
        ),
    },
    "sample_order": {
        "training": (
            "origin-major then M5 "
            "store-item series order"
        ),
        "validation": (
            "M5 store-item series order"
        ),
    },
    "locked_final_test_used": False,
}

feature_contract_payload = json.dumps(
    feature_contract,
    sort_keys=True,
    separators=(",", ":"),
).encode(
    "utf-8"
)

feature_contract_sha256 = hashlib.sha256(
    feature_contract_payload
).hexdigest()

feature_contract[
    "feature_contract_sha256"
] = feature_contract_sha256

FEATURE_CONTRACT_PATH.write_text(
    json.dumps(
        feature_contract,
        indent=2,
    ),
    encoding="utf-8",
)

1521

In [16]:
run_summary = {
    "status": "PHASE_06_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "feature_contract_sha256": (
        feature_contract_sha256
    ),
    "training_sample_count": int(
        TRAINING_SAMPLE_COUNT
    ),
    "validation_series_count": int(
        SERIES_COUNT
    ),
    "recent_lookback_days": (
        RECENT_LOOKBACK
    ),
    "recent_feature_dimension": 5,
    "decoder_feature_dimension": 128,
    "future_covariate_dimension": 8,
    "retail_shock_encoder_input_dimension": 141,
    "training_recent_shape": list(
        training_recent_features.shape
    ),
    "training_future_shape": list(
        training_future_covariates.shape
    ),
    "validation_recent_shape": list(
        validation_recent_features.shape
    ),
    "validation_future_shape": list(
        validation_future_covariates.shape
    ),
    "price_record_observed_rate": (
        price_record_observed_rate
    ),
    "price_change_valid_rate": (
        price_change_valid_rate
    ),
    "snap_rate": snap_rate,
    "event_rate": event_rate,
    "locked_final_test_used": False,
}

RUN_SUMMARY_PATH.write_text(
    json.dumps(
        run_summary,
        indent=2,
    ),
    encoding="utf-8",
)

992

In [17]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_lines = [
    "# EA-TTM Project Status",
    "",
    "## Current stage",
    (
        "Phase 06 — Retail shock feature "
        "engineering completed."
    ),
    "",
    "## Retail Shock Encoder input",
    "- Recent-demand features: 5",
    "- Horizon-aligned decoder features: 128",
    "- Future-known retail covariates: 8",
    "- Total adapter input dimension: 141",
    "",
    "## Recent-demand contract",
    "- Lookback: 28 days",
    (
        "- Features: median, robust MAD scale, "
        "7-day mean change, 28-day trend, "
        "zero-demand rate"
    ),
    "",
    "## Future-known covariates",
    (
        "- SNAP, event, price-record indicator, "
        "price-change-valid indicator, standardized "
        "log price, standardized weekly log-price "
        "change, day-of-week sine/cosine"
    ),
    "",
    "## Leakage control",
    "- Scaling fitted on training windows only",
    "- No future demand used in feature construction",
    "- Locked final-test demand not loaded",
    "",
    f"- Feature contract SHA-256: `{feature_contract_sha256}`",
    "",
    "## Next phase",
    (
        "Define the extreme-regime target and "
        "implement the Retail Shock Encoder, "
        "Extreme Gate, Correction Head, and "
        "EA-TTM residual composition."
    ),
]

status_path.write_text(
    "\n".join(
        status_lines
    ),
    encoding="utf-8",
)

982

## Phase validation

Phase 06 passes only if:

$$
\begin{aligned}
&\mathbf{z}_t
\in
\mathbb{R}^{5}
\\
&\land\
\mathbf{c}_h
\in
\mathbb{R}^{8}
\\
&\land\
\dim(
[\mathbf{z}_t;\mathbf{d}_h;\mathbf{c}_h]
)
=
141
\\
&\land\
\text{all model-ready features are finite}
\\
&\land\
\text{all scaling statistics use training data only}
\\
&\land\
\text{locked final-test demand is unused}.
\end{aligned}
$$

In [18]:
phase_validation = {
    "training_recent_shape_valid": (
        training_recent_features.shape
        == (121_960, 5)
    ),
    "training_future_shape_valid": (
        training_future_covariates.shape
        == (121_960, 28, 8)
    ),
    "validation_recent_shape_valid": (
        validation_recent_features.shape
        == (30_490, 5)
    ),
    "validation_future_shape_valid": (
        validation_future_covariates.shape
        == (30_490, 28, 8)
    ),
    "training_recent_finite": bool(
        np.isfinite(
            training_recent_features
        ).all()
    ),
    "training_future_finite": bool(
        np.isfinite(
            training_future_covariates
        ).all()
    ),
    "validation_recent_finite": bool(
        np.isfinite(
            validation_recent_features
        ).all()
    ),
    "validation_future_finite": bool(
        np.isfinite(
            validation_future_covariates
        ).all()
    ),
    "adapter_input_dimension": (
        5 + 128 + 8
    ) == 141,
    "locked_test_unused": (
        "d_1914"
        not in development_day_columns
    ),
    "feature_contract_saved": (
        FEATURE_CONTRACT_PATH.exists()
    ),
    "run_summary_saved": (
        RUN_SUMMARY_PATH.exists()
    ),
}

assert all(
    phase_validation.values()
), (
    f"Phase 06 validation failed: "
    f"{phase_validation}"
)

phase_validation

{'training_recent_shape_valid': True,
 'training_future_shape_valid': True,
 'validation_recent_shape_valid': True,
 'validation_future_shape_valid': True,
 'training_recent_finite': True,
 'training_future_finite': True,
 'validation_recent_finite': True,
 'validation_future_finite': True,
 'adapter_input_dimension': True,
 'locked_test_unused': True,
 'feature_contract_saved': True,
 'run_summary_saved': True}

## Return block

The final block is the **only successful-run output that needs to be returned to ChatGPT**.

In [19]:
return_block = {
    "status": "PHASE_06_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "feature_contract_sha256": (
        feature_contract_sha256
    ),
    "training_sample_count": int(
        TRAINING_SAMPLE_COUNT
    ),
    "validation_series_count": int(
        SERIES_COUNT
    ),
    "recent_lookback_days": (
        RECENT_LOOKBACK
    ),
    "recent_feature_names": (
        RECENT_FEATURE_NAMES
    ),
    "recent_feature_dimension": 5,
    "decoder_feature_dimension": 128,
    "future_covariate_names": (
        FUTURE_COVARIATE_NAMES
    ),
    "future_covariate_dimension": 8,
    "retail_shock_encoder_input_dimension": 141,
    "training_recent_shape": list(
        training_recent_features.shape
    ),
    "training_future_shape": list(
        training_future_covariates.shape
    ),
    "validation_recent_shape": list(
        validation_recent_features.shape
    ),
    "validation_future_shape": list(
        validation_future_covariates.shape
    ),
    "feature_rates": {
        "price_record_observed": (
            price_record_observed_rate
        ),
        "price_change_valid": (
            price_change_valid_rate
        ),
        "snap_indicator": snap_rate,
        "event_indicator": event_rate,
    },
    "locked_final_test_used": False,
    "artifacts": {
        "feature_contract": str(
            FEATURE_CONTRACT_PATH
        ),
        "feature_scaler": str(
            FEATURE_SCALER_PATH
        ),
        "leakage_audit": str(
            LEAKAGE_AUDIT_PATH
        ),
        "feature_summary": str(
            FEATURE_SUMMARY_PATH
        ),
        "train_recent_features": str(
            TRAIN_RECENT_PATH
        ),
        "validation_recent_features": str(
            VALIDATION_RECENT_PATH
        ),
        "train_future_covariates": str(
            TRAIN_FUTURE_PATH
        ),
        "validation_future_covariates": str(
            VALIDATION_FUTURE_PATH
        ),
        "run_summary": str(
            RUN_SUMMARY_PATH
        ),
        "project_status": str(
            status_path
        ),
    },
}

print("PHASE 06 RETURN BLOCK")
print(
    json.dumps(
        return_block,
        indent=2,
    )
)

PHASE 06 RETURN BLOCK
{
  "status": "PHASE_06_COMPLETE",
  "dataset_sha256": "83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59",
  "split_sha256": "5ff2e30f43a832bb9e513153bb15daa4e1e67c871b43586f2f2f11ef51396447",
  "feature_contract_sha256": "fa0abc3b4c9bf645b24e49b8f8f8582057bf3a37e7bf8fd54c374933213e01fb",
  "training_sample_count": 121960,
  "validation_series_count": 30490,
  "recent_lookback_days": 28,
  "recent_feature_names": [
    "recent_median",
    "robust_scale_mad",
    "recent_7d_mean_change",
    "short_term_28d_trend",
    "zero_demand_rate"
  ],
  "recent_feature_dimension": 5,
  "decoder_feature_dimension": 128,
  "future_covariate_names": [
    "snap_indicator",
    "event_indicator",
    "price_record_observed",
    "price_change_valid",
    "log_price",
    "weekly_log_price_change",
    "day_of_week_sin",
    "day_of_week_cos"
  ],
  "future_covariate_dimension": 8,
  "retail_shock_encoder_input_dimension": 141,
  "training_recent_shape": [
    1